# Comparación de modelos para predecir la abstención por municipio

**Objetivo.** Predecir la abstención de cada **municipio** (insumo del semáforo de riesgo para 2027) y decidir qué modelo conviene usar. Se comparan, con la **misma validación cruzada**, regresión lineal, regresión logística, gradient boosting, una red neuronal y modelos de series de tiempo (ARIMA, SARIMA, ARIMAX y SARIMAX), junto con dos reglas muy simples que sirven de piso.

**Datos.** Solo se lee `Datos/datos_procesados/base_tratada.csv`: un registro por grupo (municipio × sexo × rango de edad × tipo de sección) en cada elección. Las variables externas son las columnas de 0 y 1 (one-hot): tipo de elección, sexo, rango de edad y tipo de sección. No se usan datos externos, porcentajes de composición ni nivel nacional.

**Validación cruzada.** Ventana creciente con 3 pruebas: se entrena con lo anterior y se predice 2018, 2021 y 2024. En cada prueba se calculan las métricas sobre todos los municipios y al final se **promedian las 3 pruebas** de cada modelo. La abstención del año que se predice solo se usa para medir el error, nunca para predecir.

**Cómo se juntan los modelos.**
- Los modelos de regresión, boosting y red neuronal se entrenan **una sola vez con todos los municipios** y predicen cada grupo; después los grupos se juntan en el municipio ponderando por el tamaño del grupo (`SV + NV`).
- Los modelos de series de tiempo se ajustan **a cada municipio** con su serie de abstención de las elecciones anteriores (con 6 elecciones, solo 3 a 5 datos por ajuste). Su variable externa es el tipo de elección.

**Notas de diseño.**
- "SARIMAZ" se tomó como **SARIMAX** (SARIMA con variable externa).
- Con 6 elecciones, una serie por municipio tiene muy pocos datos. Los modelos de series de tiempo se incluyen porque se pidieron, pero se espera que sean frágiles.

## 1. Configuración y lectura
Se define la ruta de `base_tratada.csv`, las 6 elecciones y la validación cruzada (`TimeSeriesSplit` con 3 pruebas de una elección cada una), y se lee la base.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from joblib import Parallel, delayed
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

_cwd = Path.cwd().resolve()
RAIZ = next(p for p in [_cwd, *_cwd.parents] if (p / "Datos").is_dir())
RUTA_BASE = RAIZ / "Datos" / "datos_procesados" / "base_tratada.csv"

ANIOS = [2009, 2012, 2015, 2018, 2021, 2024]
PRUEBAS = list(TimeSeriesSplit(n_splits=3, test_size=1).split(ANIOS))

base = pd.read_csv(RUTA_BASE, encoding="utf-8-sig")
print(f"{len(base):,} filas | {base.shape[1]} columnas")
for entrena, prueba in PRUEBAS:
    print("Entrena con", [ANIOS[i] for i in entrena], "-> predice", ANIOS[prueba[0]])
base.head()

## 2. Municipios y grupos con las 6 elecciones
Las series de tiempo necesitan una serie sin huecos, así que se conservan solo los municipios que tienen las 6 elecciones y, dentro de ellos, los grupos (sexo × edad × tipo de sección) que aparecen en las 6. Los demás quedan sin pronóstico. Se imprime qué parte de las personas de 2024 se conserva.

In [ ]:
MUNICIPIO = ["ESTADO", "MUNICIPIO"]
COLUMNAS_GRUPO = [c for c in base.columns if c.startswith(("SEXO_", "EDAD_", "SEC_"))]
GRUPO = MUNICIPIO + COLUMNAS_GRUPO

elecciones = base.groupby(MUNICIPIO)["AELEC"].transform("nunique")
base_c = base[elecciones == len(ANIOS)]
elecciones = base_c.groupby(GRUPO)["AELEC"].transform("nunique")
base_c = base_c[elecciones == len(ANIOS)].sort_values(GRUPO + ["AELEC"]).reset_index(drop=True)

ultima = base[base["AELEC"] == 2024]
ultima_c = base_c[base_c["AELEC"] == 2024]
print(f"Municipios: {base_c.groupby(MUNICIPIO).ngroups:,} de {base.groupby(MUNICIPIO).ngroups:,}")
print(f"Grupos (municipio x sexo x edad x sección): {base_c.groupby(GRUPO).ngroups:,}")
print(f"Personas de 2024 que se conservan: {(ultima_c['SV'] + ultima_c['NV']).sum() / (ultima['SV'] + ultima['NV']).sum():.2%}")

## 3. Variables externas (one-hot) y abstención previa
Las variables externas son las columnas de 0 y 1 de la base tratada. De cada variable se quita la primera categoría (la de referencia), porque con todas las columnas sumarían siempre 1 y repetirían información. Las categorías de referencia son: elección intermedia, hombre, 18-29 años y sección mixta.

Se agregan dos columnas:
- `ANT1`: abstención del **mismo grupo** (mismo municipio, sexo, edad y sección) en la elección anterior. Es la forma de que el modelo conozca la historia del municipio.
- `ANT1_PRES`: `ANT1` multiplicada por `TIPO_Presidencial`. Sirve para que la relación con la elección anterior pueda ser distinta en presidenciales y en intermedias (la anterior de una presidencial es intermedia y al revés). El boosting y la red neuronal la podrían encontrar solos; las regresiones no.

Solo se usa la elección **anterior** (y no dos atrás) para no perder la elección de 2012 al entrenar la prueba de 2018.

In [ ]:
ONEHOT = (["TIPO_Presidencial", "SEXO_Mujer"]
          + [c for c in base_c.columns if c.startswith("EDAD_")][1:]
          + [c for c in base_c.columns if c.startswith("SEC_")][1:])

base_c["ANT1"] = base_c.groupby(GRUPO)["ABSTENCION"].shift(1)
base_c["ANT1_PRES"] = base_c["ANT1"] * base_c["TIPO_Presidencial"]
COLUMNAS_ANT = ONEHOT + ["ANT1", "ANT1_PRES"]

print(f"{len(ONEHOT)} variables one-hot:")
print(ONEHOT)
base_c[["AELEC", "ABSTENCION", "ANT1"]].describe().round(3)

## 4. Abstención por municipio, métricas y semáforo
- `serie_mun`: abstención real de cada municipio en cada elección. Se suman primero `NV` y `SV` y después se divide `NV / (SV + NV)` (nunca se promedian tasas). `serie` es la tabla municipio × elección y `real` la misma en formato largo, con las personas (`SV + NV`) de cada municipio-elección.
- `juntar`: convierte el pronóstico de cada grupo en pronóstico del municipio, ponderando por `SV + NV` (se supone conocido el tamaño de cada grupo). Cada pronóstico se recorta al rango de 0 a 1 porque la abstención es una proporción.
- `tercil`: da 1, 2 o 3 según el tercio en el que cae cada valor (1 = menor abstención). Es el semáforo provisional: verde, amarillo, rojo.
- `metricas`: para una prueba, compara pronóstico y real en **todos los municipios**:
  - **RMSE** y **MAE**: error típico en puntos de abstención (el RMSE castiga más los errores grandes).
  - **MAPE**: error relativo, en % del valor real.
  - **Sesgo**: error promedio con signo (positivo = el modelo predice más abstención de la real).
  - **MAE ponderado**: el MAE donde cada municipio pesa según sus personas; refleja el error "por persona" más que "por municipio".
  - **Acierto del semáforo**: % de municipios cuyo color pronosticado coincide con el real (al azar sería 33 %).
  - **Rojos que eran verdes**: de los municipios pronosticados en rojo, % que en realidad eran verdes. Es el error más costoso para asignar recursos.

In [ ]:
serie_mun = base_c.groupby(MUNICIPIO + ["AELEC"])[["SV", "NV"]].sum()
serie_mun["PESO"] = serie_mun["SV"] + serie_mun["NV"]
serie_mun["Real"] = serie_mun["NV"] / serie_mun["PESO"]

serie = serie_mun["Real"].unstack("AELEC")
real = serie_mun[["Real", "PESO"]].reset_index().rename(columns={"AELEC": "Año"})

def juntar(test, pron):
    t = test[MUNICIPIO].copy()
    t["PESO"] = test["SV"] + test["NV"]
    t["PRON_PESO"] = np.clip(pron, 0, 1) * t["PESO"]
    m = t.groupby(MUNICIPIO)[["PESO", "PRON_PESO"]].sum()
    return (m["PRON_PESO"] / m["PESO"]).rename("Pronóstico")

def tercil(valores):
    return np.ceil(valores.rank(pct=True) * 3)

def metricas(tabla):
    error = tabla["Pronóstico"] - tabla["Real"]
    color_real = tercil(tabla["Real"])
    color_pron = tercil(tabla["Pronóstico"])
    rojos = color_pron == 3
    return pd.Series({
        "RMSE": np.sqrt((error ** 2).mean()) * 100,
        "MAE": error.abs().mean() * 100,
        "MAPE (%)": (error.abs() / tabla["Real"]).mean() * 100,
        "Sesgo": error.mean() * 100,
        "MAE ponderado": np.average(error.abs(), weights=tabla["PESO"]) * 100,
        "Acierto semáforo (%)": (color_real == color_pron).mean() * 100,
        "Rojos que eran verdes (%)": (color_real[rojos] == 1).mean() * 100,
        "Municipios": len(tabla),
    })

print(f"Municipios: {len(serie):,} | elecciones: {list(serie.columns)}")
serie.head()

## 5. Modelos que usan todos los municipios a la vez
Cada modelo es una función que recibe la tabla de entrenamiento y la de prueba (grupos de la elección que se predice) y regresa el pronóstico de cada grupo. Todos pesan cada fila por lo que pesa su grupo (`SV + NV`, dividido entre su promedio), para que un grupo de 5 personas no cuente igual que uno de 5,000.

- **Regresión lineal: solo one-hot.** La abstención del grupo depende únicamente de tipo de elección, sexo, edad y sección. No conoce la historia del municipio, así que da lo mismo en todos los municipios de una elección (sirve de contraste).
- **Regresión lineal.** Lo anterior más `ANT1` y `ANT1_PRES`.
- **Regresión logística.** La versión de regresión pensada para proporciones: modela `NV` contra `SV` de cada grupo (distribución binomial), así que nunca da valores fuera de 0 a 100 % y respeta el tamaño de cada grupo. Se ajusta con `statsmodels`.
- **Gradient boosting.** Muchos árboles de decisión pequeños que se corrigen unos a otros; encuentra interacciones sin que se las pidamos. Las columnas son las mismas.
- **Red neuronal.** Un perceptrón multicapa (32 y 16 neuronas) con las variables escaladas. Se detiene solo cuando deja de mejorar en una parte reservada de los datos.

**Reglas de referencia** (no son modelos): copiar la abstención del municipio de la elección anterior, o la de 2 elecciones atrás (mismo tipo de elección).

In [ ]:
def peso_de(tabla):
    p = tabla["SV"] + tabla["NV"]
    return p / p.mean()

def lineal_solo_onehot(train, test):
    m = LinearRegression().fit(train[ONEHOT], train["ABSTENCION"], sample_weight=peso_de(train))
    return m.predict(test[ONEHOT])

def lineal(train, test):
    m = LinearRegression().fit(train[COLUMNAS_ANT], train["ABSTENCION"], sample_weight=peso_de(train))
    return m.predict(test[COLUMNAS_ANT])

def logistica(train, test):
    X = sm.add_constant(train[COLUMNAS_ANT], has_constant="add")
    y = np.column_stack([train["NV"], train["SV"]])
    m = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    return m.predict(sm.add_constant(test[COLUMNAS_ANT], has_constant="add"))

def boosting(train, test):
    m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, max_depth=4, random_state=0)
    m.fit(train[COLUMNAS_ANT], train["ABSTENCION"], sample_weight=peso_de(train))
    return m.predict(test[COLUMNAS_ANT])

def red_neuronal(train, test):
    escala = StandardScaler().fit(train[COLUMNAS_ANT])
    m = MLPRegressor(hidden_layer_sizes=(32, 16), early_stopping=True, max_iter=100, random_state=0)
    m.fit(escala.transform(train[COLUMNAS_ANT]), train["ABSTENCION"], sample_weight=peso_de(train))
    return m.predict(escala.transform(test[COLUMNAS_ANT]))

MODELOS_GLOBALES = {
    "Regresión lineal: solo one-hot": (lineal_solo_onehot, False),
    "Regresión lineal": (lineal, True),
    "Regresión logística": (logistica, True),
    "Gradient boosting": (boosting, True),
    "Red neuronal": (red_neuronal, True),
}
# El segundo valor indica si el modelo necesita ANT1 (entonces se entrena solo con elecciones que la tienen).

REGLAS = {
    "Regla: copiar la elección previa": 1,
    "Regla: copiar la de 2 elecciones atrás": 2,
}

## 6. Validación cruzada de los modelos globales y de las reglas
Para cada una de las 3 pruebas se separa lo que se conoce (elecciones anteriores) de la elección que se predice, se entrena cada modelo, se predicen los grupos de esa elección y se juntan en municipio. Las reglas se calculan directo de la tabla `serie`. Puede tardar unos minutos (la red neuronal y el boosting son los más lentos).

In [ ]:
bloques = []
for prueba, (entrena, test_i) in enumerate(PRUEBAS, start=1):
    t = test_i[0]
    antes = base_c[base_c["AELEC"].isin([ANIOS[i] for i in entrena])]
    test = base_c[base_c["AELEC"] == ANIOS[t]]

    for nombre, (funcion, usa_ant) in MODELOS_GLOBALES.items():
        train = antes.dropna(subset=["ANT1"]) if usa_ant else antes
        pron = juntar(test, funcion(train, test)).reset_index()
        pron["Modelo"], pron["Prueba"], pron["Año"] = nombre, prueba, ANIOS[t]
        bloques.append(pron)
        print(f"Prueba {prueba} ({ANIOS[t]}): {nombre}")

    for nombre, atras in REGLAS.items():
        pron = serie[ANIOS[t - atras]].rename("Pronóstico").reset_index()
        pron["Modelo"], pron["Prueba"], pron["Año"] = nombre, prueba, ANIOS[t]
        bloques.append(pron)

pronosticos_globales = pd.concat(bloques, ignore_index=True)
print(f"\n{len(pronosticos_globales):,} filas (municipio x modelo x prueba)")
pronosticos_globales.head()

## 7. Modelos de series de tiempo (uno por municipio)
Para cada municipio la serie es su abstención en las 6 elecciones. En cada prueba se ajusta el modelo con las elecciones anteriores (3, 4 y 5 datos) y se pronostica la siguiente. La variable externa es el tipo de elección (`TIPO_Presidencial`: 0 = intermedia, 1 = presidencial), que se conoce de antemano.

- **ARIMA(1,0,0):** la abstención de hoy depende de la anterior (con una constante).
- **ARIMA(1,1,0):** igual, pero sobre los *cambios* entre elecciones (diferencia).
- **SARIMA(1,0,0)(1,0,0,2):** agrega un componente "estacional" de periodo 2, porque las elecciones alternan presidencial e intermedia.
- **ARIMAX(1,0,0) y (1,1,0):** los ARIMA anteriores con el tipo de elección como variable externa.
- **SARIMAX(1,0,0)(1,0,0,2):** SARIMA con el tipo de elección como variable externa. Como el tipo de elección ya alterna con periodo 2, repite parte de la información del componente estacional.

Si un ajuste falla (con tan pocos datos pasa), ese municipio queda sin pronóstico en ese modelo; se cuenta más adelante. Los ajustes se reparten entre los núcleos de la computadora con `joblib`.

In [ ]:
TIPO = np.array([0, 1, 0, 1, 0, 1])  # 2009, 2012, ..., 2024 (1 = presidencial)

MODELOS_TIEMPO = {
    "ARIMA(1,0,0)": dict(order=(1, 0, 0), con_tipo=False),
    "ARIMA(1,1,0)": dict(order=(1, 1, 0), con_tipo=False),
    "SARIMA(1,0,0)(1,0,0,2)": dict(order=(1, 0, 0), seasonal_order=(1, 0, 0, 2), con_tipo=False),
    "ARIMAX(1,0,0)": dict(order=(1, 0, 0), con_tipo=True),
    "ARIMAX(1,1,0)": dict(order=(1, 1, 0), con_tipo=True),
    "SARIMAX(1,0,0)(1,0,0,2)": dict(order=(1, 0, 0), seasonal_order=(1, 0, 0, 2), con_tipo=True),
}

def pronosticar(y, t, configuracion):
    warnings.filterwarnings("ignore")  # cada proceso paralelo necesita su propio filtro
    config = dict(configuracion)
    con_tipo = config.pop("con_tipo")
    exog = TIPO.reshape(-1, 1).astype(float) if con_tipo else None
    try:
        modelo = SARIMAX(y[:t], exog=exog[:t] if con_tipo else None,
                         trend="c" if config["order"][1] == 0 else "n", **config)
        ajuste = modelo.fit(disp=False)
        return float(ajuste.forecast(steps=1, exog=exog[t:t + 1] if con_tipo else None)[0])
    except Exception:
        return np.nan

def ajustar_municipio(estado, municipio, y):
    filas = []
    for prueba, (entrena, test_i) in enumerate(PRUEBAS, start=1):
        t = test_i[0]
        for nombre, configuracion in MODELOS_TIEMPO.items():
            filas.append((estado, municipio, nombre, prueba, ANIOS[t], pronosticar(y, t, configuracion)))
    return filas

# Prueba rápida con un municipio antes de lanzar todos
estado_ej, municipio_ej = serie.index[0]
print(estado_ej, "-", municipio_ej, "| serie real:", serie.iloc[0].round(3).tolist())
pd.DataFrame(ajustar_municipio(estado_ej, municipio_ej, serie.iloc[0].to_numpy()),
             columns=MUNICIPIO + ["Modelo", "Prueba", "Año", "Pronóstico"]).round(3)

In [ ]:
resultado = Parallel(n_jobs=-1, batch_size=20)(
    delayed(ajustar_municipio)(estado, municipio, y)
    for (estado, municipio), y in zip(serie.index, serie.to_numpy())
)

pronosticos_tiempo = pd.DataFrame([f for fila in resultado for f in fila],
                                  columns=MUNICIPIO + ["Modelo", "Prueba", "Año", "Pronóstico"])
print(f"{len(pronosticos_tiempo):,} filas (municipio x modelo x prueba)")
print("Pronósticos que fallaron, por modelo:")
print(pronosticos_tiempo[pronosticos_tiempo["Pronóstico"].isna()].groupby("Modelo").size().rename("Sin pronóstico"))

## 8. Métricas de cada prueba
Se juntan todos los pronósticos con la abstención real y las personas de cada municipio-elección, y se calculan las métricas **por modelo y por prueba** (cada una sobre todos los municipios). Los pronósticos de series de tiempo fuera de [0, 1] se recortan igual que los demás y los que fallaron se excluyen de las métricas de ese modelo (la columna `Municipios` dice con cuántos se calculó).

In [ ]:
pronosticos = pd.concat([pronosticos_globales, pronosticos_tiempo], ignore_index=True)
pronosticos["Pronóstico"] = pronosticos["Pronóstico"].clip(0, 1)
pronosticos = pronosticos.merge(real, on=MUNICIPIO + ["Año"], how="left")

filas = []
for (modelo, prueba), tabla in pronosticos.dropna(subset=["Pronóstico"]).groupby(["Modelo", "Prueba"], sort=False):
    filas.append(metricas(tabla).rename((modelo, prueba)))
por_prueba = pd.DataFrame(filas)
por_prueba.index.names = ["Modelo", "Prueba"]
por_prueba["Año"] = [ANIOS[PRUEBAS[p - 1][1][0]] for _, p in por_prueba.index]
por_prueba.round(2)

## 9. Promedio de las 3 pruebas por modelo
Para cada modelo se promedian sus métricas de las 3 pruebas. `Pruebas` es en cuántas se pudo calcular. La tabla se ordena de menor a mayor RMSE.

In [ ]:
columnas = ["RMSE", "MAE", "MAPE (%)", "Sesgo", "MAE ponderado", "Acierto semáforo (%)", "Rojos que eran verdes (%)"]
promedio = por_prueba.groupby("Modelo")[columnas].mean()
promedio["Municipios (prom.)"] = por_prueba.groupby("Modelo")["Municipios"].mean()
promedio["Pruebas"] = por_prueba.groupby("Modelo").size()
promedio = promedio.sort_values("RMSE")
promedio.round(2)

## 10. Comparación final
Tres vistas: el RMSE y el acierto del semáforo promedio de cada modelo, y el RMSE en cada prueba (para ver si un modelo solo gana en un año).

In [ ]:
orden = promedio.index.tolist()
colores = ["tab:gray" if m.startswith("Regla") else "tab:blue" for m in orden]

fig, ejes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
ejes[0].barh(orden, promedio["RMSE"], color=colores)
ejes[0].set_title("RMSE promedio (puntos de abstención, menor es mejor)")
ejes[1].barh(orden, promedio["Acierto semáforo (%)"], color=colores)
ejes[1].axvline(100 / 3, color="red", linestyle="--", label="Azar (33 %)")
ejes[1].set_title("Acierto del semáforo promedio (%, mayor es mejor)")
ejes[1].legend()
for eje in ejes:
    eje.invert_yaxis()
plt.tight_layout()
plt.show()

rmse_prueba = por_prueba["RMSE"].unstack("Prueba").loc[orden]
rmse_prueba.columns = [f"Prueba {p} ({ANIOS[PRUEBAS[p - 1][1][0]]})" for p in rmse_prueba.columns]
rmse_prueba.round(2)

In [ ]:
mejor_rmse = promedio["RMSE"].idxmin()
mejor_semaforo = promedio["Acierto semáforo (%)"].idxmax()
mejor_sin_reglas = promedio[~promedio.index.str.startswith("Regla")]["RMSE"].idxmin()
print(f"Menor RMSE promedio: {mejor_rmse} ({promedio.loc[mejor_rmse, 'RMSE']:.2f} puntos)")
print(f"Mayor acierto del semáforo: {mejor_semaforo} ({promedio.loc[mejor_semaforo, 'Acierto semáforo (%)']:.1f} %)")
print(f"Mejor modelo (sin contar reglas): {mejor_sin_reglas} ({promedio.loc[mejor_sin_reglas, 'RMSE']:.2f} puntos)")
mejores_reglas = promedio[promedio.index.str.startswith("Regla")]["RMSE"].min()
print(f"Mejor regla simple: {mejores_reglas:.2f} puntos")